# Notebook 5: MIPA-specific YOLOv8 Finetune
**ANPR System – MIPA UGM Parking Lot**

Single-purpose finetune of `yolov8s.pt` on photos from the MIPA gate, so the
deployed model has high confidence on *your* lighting, angles, and distances
instead of generic Indonesian-plate conditions.

Prerequisites:
1. You've collected ~200–300 photos per `docs/TRAINING.md`
2. You've labeled them on Roboflow with tight bounding boxes around plate text only
3. You've run `scripts/prep_mipa_dataset.py` locally and uploaded the resulting
   folder to Drive at `/MyDrive/ANPR_MIPA_UGM/datasets/mipa_v1/`

Expected runtime on a Colab T4: **30–60 minutes** for 50 epochs at imgsz 1280.

Expected outcome: mAP50 ≥ 0.90 on the validation set, MIPA-deployment plates
detecting at 0.85+ confidence on the spec's default thresholds.

## 1. Setup

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('!! No GPU detected. Change runtime → T4 GPU in Colab.')

DRIVE_ROOT = '/content/drive/MyDrive/ANPR_MIPA_UGM'
DATASET_NAME = 'mipa_v1'  # change to mipa_v2, mipa_v3, ... for re-runs
DATA_YAML = f'{DRIVE_ROOT}/datasets/{DATASET_NAME}/data.yaml'
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'
RUN_NAME = f'yolov8s_{DATASET_NAME}'

import os
os.makedirs(WEIGHTS_DIR, exist_ok=True)
assert os.path.exists(DATA_YAML), f'data.yaml not found at {DATA_YAML} — did you upload the dataset?'
print(f'\nUsing: {DATA_YAML}')

## 2. Sanity-check the dataset

Before training, draw the labels on a few training images. If the boxes
don't tightly hug the plate text, your finetune will inherit those
loose-bbox problems and the dealer-text / sticker bug returns.

In [ ]:
import cv2, yaml
import matplotlib.pyplot as plt
from pathlib import Path

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
dataset_root = Path(cfg['path'])
train_img_dir = dataset_root / cfg['train']
train_lbl_dir = dataset_root / cfg['train'].replace('images', 'labels')

samples = sorted(train_img_dir.glob('*'))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                _, xc, yc, bw, bh = map(float, parts)
                x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(img); ax.axis('off'); ax.set_title(img_path.name[:24], fontsize=8)
plt.suptitle('Sanity check: labels should hug ONLY the plate text', fontsize=14)
plt.tight_layout(); plt.show()

## 3. Configure the finetune

Hyperparameters chosen for **transfer learning**, not from-scratch training:

- `lr0=0.0005` — half the spec's 0.001. We're starting from a good base; high
  LR would destroy the existing weights.
- `epochs=50` — enough for transfer learning on 200–300 images. The base
  notebook (`02_train_yolov8.ipynb`) uses 100 for from-scratch.
- `imgsz=1280` — matches our inference letterbox. Training at 640 and inferring
  at 1280 wastes the extra resolution.
- `patience=15` — stop early if validation mAP plateaus.

In [ ]:
TRAIN_ARGS = {
    'data': DATA_YAML,
    'epochs': 50,
    'batch': 8,            # 1280 imgsz needs less batch than 640. Drop to 4 if OOM.
    'imgsz': 1280,
    'optimizer': 'AdamW',
    'lr0': 0.0005,
    'lrf': 0.01,
    'cos_lr': True,
    'patience': 15,
    'save': True,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 4,
    'cache': True,
    'plots': True,
    # Augmentations: keep mild — Roboflow already applied the heavy ones
    'hsv_h': 0.01, 'hsv_s': 0.5, 'hsv_v': 0.3,
    'degrees': 5.0, 'translate': 0.05, 'scale': 0.2,
    'fliplr': 0.0,         # plates are not symmetric! disable horizontal flip
    'mosaic': 0.5, 'mixup': 0.0,
}
print('Training args:')
for k, v in TRAIN_ARGS.items():
    print(f'  {k:<12} = {v}')

## 4. Train

In [ ]:
from ultralytics import YOLO

# Start from the general-purpose yolov8s weights (Ultralytics auto-downloads on first use)
model = YOLO('yolov8s.pt')

results = model.train(
    project=f'{DRIVE_ROOT}/runs',
    name=RUN_NAME,
    **TRAIN_ARGS,
)
best_pt = f'{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt'
print(f'\nBest weights: {best_pt}')

## 5. Validate on the test split

In [ ]:
trained = YOLO(best_pt)
metrics = trained.val(data=DATA_YAML, split='test', imgsz=1280, plots=True)
print(f'\nTest-set results:')
print(f'  mAP50     = {metrics.box.map50:.4f}   (target: > 0.90)')
print(f'  mAP50-95  = {metrics.box.map:.4f}   (target: > 0.65)')
print(f'  precision = {metrics.box.mp:.4f}')
print(f'  recall    = {metrics.box.mr:.4f}')

## 6. Save weights for download

After this cell finishes, download `best_yolov8s_mipa.pt` from your Drive's
`weights/` folder and drop it into the repo at `models/best_yolov8s.pt`
(overwriting the existing one). Then restore the spec defaults in
`config.yaml` (`confidence_threshold: 0.5`, `input_size: [640, 640]`).

In [ ]:
import shutil, os
out = f'{WEIGHTS_DIR}/best_yolov8s_mipa.pt'
shutil.copy2(best_pt, out)
size_mb = os.path.getsize(out) / 1e6
print(f'Saved {out} ({size_mb:.1f} MB)')
print('\nNext steps:')
print('  1. Download best_yolov8s_mipa.pt from Drive')
print('  2. Locally: mv ~/Downloads/best_yolov8s_mipa.pt models/best_yolov8s.pt')
print('  3. In config.yaml: confidence_threshold: 0.5, input_size: [640, 640]')
print('  4. Flask auto-reloads; verify via /healthz')
print('  5. Re-upload your test images — expect 0.85+ det conf')

## 7. Optional — predict on a few test images here in Colab

Quick visual confidence-check before downloading. If everything looks good in
these plots, the model is ready for deployment.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt, cv2, numpy as np

test_img_dir = dataset_root / 'test' / 'images'
samples = sorted(test_img_dir.glob('*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, p in zip(axes.flat, samples):
    res = trained.predict(str(p), conf=0.25, imgsz=1280, verbose=False)[0]
    img = res.plot()  # ultralytics renders bbox + label + conf
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    ax.set_title(f'{p.name}\n{len(res.boxes)} plate(s)', fontsize=10)
plt.tight_layout(); plt.show()